# C_S3.0 — Permutation Test Result Analysis

Analyzes the output of `C_S3_permutation.ipynb` (`permutation_all.parquet`).

1. **Overview** — classification distribution, false positive / false negative rates
2. **By Category** — true_null vs variance_only vs mean_only vs mean+variance
3. **By Family** — per-function detection rates across SNR
4. **Metric Power** — which z-score metrics drive detection
5. **Joint Test Diagnostics** — T_joint and p-value distributions

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

S1_DIR  = Path('output/S1')
S3_DIR  = Path('output/S3')

In [ ]:
# ── Load S3 results + S1 case metadata ──
perm = pd.read_parquet(S3_DIR / 'permutation_all.parquet')

cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_main['source'] = 'main'
cases_null['source'] = 'null_expanded'
offset = cases_main['case_id'].max()
cases_null['case_id'] = cases_null['case_id'] + offset
cases_all = pd.concat([cases_main, cases_null], ignore_index=True)

meta_cols = ['case_id', 'family_id', 'family_name', 'snr', 'spread_pattern',
             'x_distribution', 'category', 'direction', 'linearity',
             'monotonicity', 'special_shape', 'basic_label']
meta = cases_all[[c for c in meta_cols if c in cases_all.columns]].copy()

df = perm.merge(meta, on='case_id', how='left', suffixes=('', '_meta'))
if 'category_meta' in df.columns:
    df['category'] = df['category_meta']
    df.drop(columns='category_meta', inplace=True)

z_cols = [c for c in df.columns if c.startswith('z_')]
p_cols = [c for c in df.columns if c.startswith('p_') and c != 'p_value']

print(f'Loaded: {len(df):,} cases × {len(df.columns)} columns')
print(f'Z-score metrics: {len(z_cols)}')
print(f'\nClassification:\n{df["classification"].value_counts().to_string()}')
print(f'\nCategory:\n{df["category"].value_counts().to_string()}')

## 1. Overview

In [ ]:
# ── Classification × Category crosstab ──
ct = pd.crosstab(df['category'], df['classification'], margins=True)
ct_pct = pd.crosstab(df['category'], df['classification'], normalize='index').mul(100).round(1)

print('=== Counts ===')
print(ct.to_string())
print('\n=== Row % ===')
print(ct_pct.to_string())

# Key rates — only true_null is genuinely null (no relationship)
# variance_only has a variance-x relationship and IS detectable
null_cats = ['true_null']
signal_cats = ['mean_only', 'variance_only', 'mean+variance']

null_df = df[df['category'].isin(null_cats)]
signal_df = df[df['category'].isin(signal_cats)]

fpr = (null_df['classification'] == 'detectable').mean()
fnr = (signal_df['classification'] == 'not_detectable').mean()

print(f'\n── Key Rates ──')
print(f'False Positive Rate (true_null → detectable):  {fpr:.3%}')
print(f'False Negative Rate (signal → not_detectable): {fnr:.3%}')
print(f'Uncertain rate (all):                          {(df["classification"]=="uncertain").mean():.3%}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9),
                         gridspec_kw={'height_ratios': [3, 1, 0.6], 'hspace': 0.05})

# ── Classification bar by category ──
cat_order = ['true_null', 'variance_only', 'mean_only', 'mean+variance']
cls_order = ['detectable', 'uncertain', 'not_detectable']
colors = {'detectable': '#2ca02c', 'uncertain': '#ff7f0e', 'not_detectable': '#d62728'}

ct_plot = pd.crosstab(df['category'], df['classification'], normalize='index')
ct_plot = ct_plot.reindex(index=cat_order, columns=cls_order, fill_value=0)
ct_plot.plot.bar(stacked=True, ax=axes[0], color=[colors[c] for c in cls_order],
                 edgecolor='white', linewidth=0.5, legend=False)
axes[0].set_title('Classification by Category')
axes[0].set_ylabel('Proportion')
axes[0].set_xlabel('')
axes[0].set_xticklabels([])
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].legend(loc='upper right', fontsize=9)

# ── Table 1: per category ──
axes[1].axis('off')
ct_counts = pd.crosstab(df['category'], df['classification'])
ct_counts = ct_counts.reindex(index=cat_order, columns=cls_order, fill_value=0)
ct_pct = (ct_plot * 100).round(1)

cell_text = []
for cat in cat_order:
    n_cat = ct_counts.loc[cat].sum()
    row = [cat, str(n_cat)]
    for cls in cls_order:
        cnt = ct_counts.loc[cat, cls]
        pct = ct_pct.loc[cat, cls]
        row.append(f'{cnt:,}  ({pct:.1f}%)')
    cell_text.append(row)

col_labels = ['Category', 'n'] + [c for c in cls_order]
tbl1 = axes[1].table(cellText=cell_text, colLabels=col_labels,
                      loc='center', cellLoc='center')
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(10)
tbl1.scale(1, 1.5)

for (row, col), cell in tbl1.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if row == 0:
        cell.set_facecolor('#e8e8e8')
        cell.set_text_props(fontweight='bold')
    elif col >= 2:
        cell.set_facecolor(colors[cls_order[col - 2]] + '18')

# ── Table 2: Has relationship / No relationship ──
axes[2].axis('off')

groups = {'Has relationship': ['mean_only', 'variance_only', 'mean+variance'],
          'No relationship': ['true_null']}
cell_text2 = []
for label, cats in groups.items():
    mask = df['category'].isin(cats)
    n_grp = mask.sum()
    row = [label, str(n_grp)]
    for cls in cls_order:
        cnt = (df.loc[mask, 'classification'] == cls).sum()
        pct = cnt / n_grp * 100
        row.append(f'{cnt:,}  ({pct:.1f}%)')
    cell_text2.append(row)

tbl2 = axes[2].table(cellText=cell_text2, colLabels=col_labels,
                      loc='center', cellLoc='center')
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(10)
tbl2.scale(1, 1.5)

for (row, col), cell in tbl2.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if row == 0:
        cell.set_facecolor('#e8e8e8')
        cell.set_text_props(fontweight='bold')
    elif col >= 2:
        cell.set_facecolor(colors[cls_order[col - 2]] + '18')

plt.tight_layout()
plt.show()

### Binary decision (p ≤ α)

Standard hypothesis test: single threshold α = 0.05, no uncertain zone.

In [ ]:
# ── Binary classification: p ≤ 0.05 → detected, else not detected ──
ALPHA = 0.05
df['binary_cls'] = np.where(df['p_value'] <= ALPHA, 'detected', 'not_detected')

cat_order = ['true_null', 'variance_only', 'mean_only', 'mean+variance']
bin_order = ['detected', 'not_detected']
colors_bin = {'detected': '#4a90d9', 'not_detected': '#b0b0b0'}

fig, axes = plt.subplots(3, 1, figsize=(10, 9),
                         gridspec_kw={'height_ratios': [3, 0.6, 1], 'hspace': 0.05})

# ── Bar chart ──
ct_bin = pd.crosstab(df['category'], df['binary_cls'], normalize='index')
ct_bin = ct_bin.reindex(index=cat_order, columns=bin_order, fill_value=0)
ct_bin.plot.bar(stacked=True, ax=axes[0],
                color=[colors_bin[c] for c in bin_order],
                edgecolor='white', linewidth=0.5, legend=False)
axes[0].set_title(f'Binary Classification (α = {ALPHA})')
axes[0].set_ylabel('Proportion')
axes[0].set_xlabel('')
axes[0].set_xticklabels([])
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].legend(loc='upper right', fontsize=9)

col_labels = ['Category', 'n'] + bin_order

# ── Table 1: Has relationship / No relationship (2 rows) — top ──
axes[1].axis('off')
groups = {'Has relationship': ['mean_only', 'variance_only', 'mean+variance'],
          'No relationship': ['true_null']}
cell_text_grp = []
for label, cats in groups.items():
    mask = df['category'].isin(cats)
    n_grp = mask.sum()
    row = [label, str(n_grp)]
    for cls in bin_order:
        cnt = (df.loc[mask, 'binary_cls'] == cls).sum()
        pct = cnt / n_grp * 100
        row.append(f'{cnt:,}  ({pct:.1f}%)')
    cell_text_grp.append(row)

tbl1 = axes[1].table(cellText=cell_text_grp, colLabels=col_labels,
                      loc='center', cellLoc='center')
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(10)
tbl1.scale(1, 1.5)
for (row, col), cell in tbl1.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if row == 0:
        cell.set_facecolor('#e8e8e8')
        cell.set_text_props(fontweight='bold')
    elif col >= 2:
        cell.set_facecolor(colors_bin[bin_order[col - 2]] + '18')

# ── Table 2: per category (4 rows) — bottom ──
axes[2].axis('off')
ct_counts = pd.crosstab(df['category'], df['binary_cls'])
ct_counts = ct_counts.reindex(index=cat_order, columns=bin_order, fill_value=0)
ct_pct = (ct_bin * 100).round(1)

cell_text_cat = []
for cat in cat_order:
    n_cat = ct_counts.loc[cat].sum()
    row = [cat, str(n_cat)]
    for cls in bin_order:
        cnt = ct_counts.loc[cat, cls]
        pct = ct_pct.loc[cat, cls]
        row.append(f'{cnt:,}  ({pct:.1f}%)')
    cell_text_cat.append(row)

tbl2 = axes[2].table(cellText=cell_text_cat, colLabels=col_labels,
                      loc='center', cellLoc='center')
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(10)
tbl2.scale(1, 1.5)
for (row, col), cell in tbl2.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if row == 0:
        cell.set_facecolor('#e8e8e8')
        cell.set_text_props(fontweight='bold')
    elif col >= 2:
        cell.set_facecolor(colors_bin[bin_order[col - 2]] + '18')

plt.tight_layout()
plt.show()

# ── Compare: where did the uncertain cases go? ──
n_unc = (df['classification'] == 'uncertain').sum()
unc_to_det = ((df['classification'] == 'uncertain') & (df['binary_cls'] == 'detected')).sum()
unc_to_nd = ((df['classification'] == 'uncertain') & (df['binary_cls'] == 'not_detected')).sum()
print(f'Three-class uncertain cases: {n_unc}')
print(f'  → binary detected:     {unc_to_det} ({unc_to_det/n_unc:.1%})')
print(f'  → binary not_detected: {unc_to_nd} ({unc_to_nd/n_unc:.1%})')

## 2. By Category — p-value and T_joint distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

cat_order = ['true_null', 'variance_only', 'mean_only', 'mean+variance']
cat_colors = {'true_null': '#1f77b4', 'variance_only': '#ff7f0e',
              'mean_only': '#2ca02c', 'mean+variance': '#d62728'}

# 2a: p-value by category (log scale)
for cat in cat_order:
    sub = df[df['category'] == cat]
    axes[0].hist(sub['p_value'], bins=50, alpha=0.5, label=f'{cat} (n={len(sub)})',
                 color=cat_colors[cat], density=True)
axes[0].axvline(0.05, color='red', ls='--', lw=1, alpha=0.7)
axes[0].set_title('p-value Distribution by Category')
axes[0].set_xlabel('p-value')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)

# 2b: T_joint boxplot
data_box = [df[df['category'] == cat]['T_joint'].dropna() for cat in cat_order]
bp = axes[1].boxplot(data_box, labels=cat_order, patch_artist=True,
                     showfliers=False, widths=0.6)
for patch, cat in zip(bp['boxes'], cat_order):
    patch.set_facecolor(cat_colors[cat])
    patch.set_alpha(0.6)
axes[1].set_title('T_joint (max Z-score) by Category')
axes[1].set_ylabel('T_joint')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ── Null calibration: p-value should be uniform under true null ──
null_pvals = df[df['category'] == 'true_null']['p_value']

fig, ax = plt.subplots(figsize=(5, 5))
n_null = len(null_pvals)
expected = np.linspace(0, 1, n_null + 2)[1:-1]
observed = np.sort(null_pvals.values)
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Uniform (ideal)')
ax.plot(expected, observed, '.', markersize=2, alpha=0.5, color='steelblue')
ax.set_xlabel('Expected (Uniform)')
ax.set_ylabel('Observed p-value')
ax.set_title(f'QQ-plot: true_null p-values (n={n_null})')
ax.set_aspect('equal')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'true_null: {(null_pvals <= 0.05).mean():.1%} have p ≤ 0.05 (expected ~5%)')
print(f'true_null: {(null_pvals <= 0.10).mean():.1%} have p ≤ 0.10 (expected ~10%)')

## 3. By Family — Detection Rate vs SNR

In [ ]:
# ── Detection rate heatmap: family × SNR ──
signal = df[df['category'].isin(['mean_only', 'mean+variance'])].copy()

det_rate = signal.groupby(['family_id', 'snr']).apply(
    lambda g: (g['classification'] == 'detectable').mean()
).unstack('snr')

# Sort SNR columns numerically
snr_order = sorted(det_rate.columns, key=lambda x: float(x) if np.isfinite(float(x)) else 1e6)
det_rate = det_rate[snr_order]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(det_rate, annot=True, fmt='.0%', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Detection Rate'})
ax.set_title('Detection Rate by Family × SNR (signal cases only)')
ax.set_xlabel('SNR')
ax.set_ylabel('Family')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-family summary ──
fam_summary = signal.groupby('family_id').agg(
    n=('classification', 'size'),
    det_rate=('classification', lambda x: (x == 'detectable').mean()),
    median_T=('T_joint', 'median'),
    median_p=('p_value', 'median'),
    min_snr_detected=('snr', lambda x: x[signal.loc[x.index, 'classification'] == 'detectable'].min()
                      if (signal.loc[x.index, 'classification'] == 'detectable').any() else np.inf)
).sort_values('det_rate')

fam_summary['det_rate'] = fam_summary['det_rate'].map('{:.1%}'.format)
fam_summary['median_p'] = fam_summary['median_p'].map('{:.4f}'.format)
fam_summary['median_T'] = fam_summary['median_T'].map('{:.1f}'.format)

print('=== Per-Family Detection Summary (signal cases, sorted by det_rate) ===')
print(fam_summary.to_string())

In [ ]:
# ── Detection curve: detection rate vs SNR for selected families ──
det_by_snr = signal.groupby(['family_id', 'snr']).apply(
    lambda g: (g['classification'] == 'detectable').mean()
).reset_index(name='det_rate')

# Pick families with interesting patterns (low overall detection)
overall_det = signal.groupby('family_id')['classification'].apply(
    lambda x: (x == 'detectable').mean()
).sort_values()

hard_families = overall_det[overall_det < 0.95].index.tolist()
easy_families = overall_det[overall_det >= 0.95].index.tolist()

if hard_families:
    fig, ax = plt.subplots(figsize=(10, 5))
    for fam in hard_families:
        sub = det_by_snr[det_by_snr['family_id'] == fam].sort_values('snr')
        finite_mask = np.isfinite(sub['snr'])
        ax.plot(sub.loc[finite_mask, 'snr'], sub.loc[finite_mask, 'det_rate'],
                'o-', label=fam, markersize=5)
    ax.set_xscale('log')
    ax.set_xlabel('SNR')
    ax.set_ylabel('Detection Rate')
    ax.set_title(f'Detection Rate vs SNR — harder families (det < 95%)')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.8, color='gray', ls=':', lw=1, alpha=0.5)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print('All families have >95% detection rate')

print(f'\nEasy families (≥95% detected): {easy_families}')
print(f'Hard families (<95% detected): {hard_families}')

## 4. By Spread Pattern

In [ ]:
# ── Detection rate by spread_pattern ──
spread_ct = pd.crosstab(df['spread_pattern'], df['classification'], normalize='index').mul(100)
print('Detection by spread_pattern (%):')
print(spread_ct.round(1).to_string())

# variance_only: detection rate by spread pattern (these ARE real detections, not FP)
vo = df[df['category'] == 'variance_only'].copy()
if len(vo) > 0:
    print(f'\nvariance_only detection rate by spread_pattern:')
    det_by_spread = vo.groupby('spread_pattern')['classification'].apply(
        lambda x: f"{(x=='detectable').mean():.1%}  (n={len(x)})")
    print(det_by_spread.to_string())

## 5. Metric Power — Which metrics drive detection?

In [ ]:
# ── For each detectable case, which metric gave the highest z-score? ──
det = df[df['classification'] == 'detectable'].copy()
z_valid = [c for c in z_cols if det[c].notna().sum() > 0]

best_metric = det[z_valid].idxmax(axis=1).str.replace('z_', '', regex=False)
top_drivers = best_metric.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_drivers.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('# cases where this metric had max Z')
ax.set_title('Top detection-driving metrics (detectable cases)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\nTop 10 drivers (% of {len(det):,} detectable cases):')
for metric, count in top_drivers.head(10).items():
    print(f'  {metric:30s}  {count:5d}  ({count/len(det):.1%})')

In [ ]:
# ── Z-score separation: signal vs null for top metrics ──
is_null = df['category'] == 'true_null'
is_signal = df['category'].isin(['mean_only', 'variance_only', 'mean+variance'])

metric_sep = []
for zc in z_cols:
    vals_null = df.loc[is_null, zc].dropna()
    vals_sig = df.loc[is_signal, zc].dropna()
    if len(vals_null) > 0 and len(vals_sig) > 0:
        sep = vals_sig.median() - vals_null.median()
        metric_sep.append({'metric': zc.replace('z_', ''), 'separation': sep,
                           'null_med': vals_null.median(), 'signal_med': vals_sig.median()})

sep_df = pd.DataFrame(metric_sep).sort_values('separation', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_n = min(25, len(sep_df))
top_sep = sep_df.head(top_n)
y_pos = range(top_n)
ax.barh(y_pos, top_sep['separation'], color='steelblue', edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_sep['metric'], fontsize=9)
ax.set_xlabel('Median Z (signal) − Median Z (null)')
ax.set_title('Metric Discriminative Power (Z-score separation)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Metric correlation (signal cases only): identify redundant metrics ──
z_data = df.loc[is_signal, z_valid].dropna(axis=1, how='all')
corr = z_data.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.3, ax=ax,
            xticklabels=[c.replace('z_', '') for c in corr.columns],
            yticklabels=[c.replace('z_', '') for c in corr.columns],
            cbar_kws={'shrink': 0.6})
ax.set_title('Z-score Correlation (signal cases)')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

# Highly correlated pairs (|r| > 0.95)
high_corr = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        r = corr.iloc[i, j]
        if abs(r) > 0.95:
            high_corr.append((corr.index[i].replace('z_',''),
                              corr.columns[j].replace('z_',''), f'{r:.3f}'))

if high_corr:
    print(f'Highly correlated pairs (|r| > 0.95): {len(high_corr)}')
    for a, b, r in high_corr[:15]:
        print(f'  {a:30s} ↔ {b:30s}  r={r}')
    if len(high_corr) > 15:
        print(f'  ... and {len(high_corr)-15} more')
else:
    print('No pairs with |r| > 0.95')

## 6. Joint Test Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 6a: T_joint vs p_value scatter
for cat, color in cat_colors.items():
    sub = df[df['category'] == cat]
    axes[0].scatter(sub['T_joint'], sub['p_value'], s=3, alpha=0.3,
                    color=color, label=cat)
axes[0].axhline(0.05, color='red', ls='--', lw=1, alpha=0.5)
axes[0].set_xlabel('T_joint (max Z)')
axes[0].set_ylabel('p-value')
axes[0].set_title('T_joint vs p-value')
axes[0].legend(fontsize=8, markerscale=3)

# 6b: T_joint histogram by classification
for cls, color in colors.items():
    sub = df[df['classification'] == cls]
    axes[1].hist(sub['T_joint'], bins=60, alpha=0.5, label=f'{cls} (n={len(sub)})',
                 color=color, density=True)
axes[1].set_xlabel('T_joint')
axes[1].set_ylabel('Density')
axes[1].set_title('T_joint Distribution by Classification')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Misclassified cases: false positives and false negatives ──
fp = df[(df['category'].isin(null_cats)) & (df['classification'] == 'detectable')]
fn = df[(df['category'].isin(signal_cats)) & (df['classification'] == 'not_detectable')]

print(f'=== False Positives: {len(fp)} null cases classified as detectable ===')
if len(fp) > 0:
    print(fp.groupby(['category', 'spread_pattern']).size().to_string())
    print(f'\nFP T_joint range: [{fp["T_joint"].min():.2f}, {fp["T_joint"].max():.2f}]')
    print(f'FP p-value range: [{fp["p_value"].min():.4f}, {fp["p_value"].max():.4f}]')

print(f'\n=== False Negatives: {len(fn)} signal cases classified as not_detectable ===')
if len(fn) > 0:
    fn_by_fam = fn.groupby('family_id').agg(
        n=('case_id', 'size'),
        snr_range=('snr', lambda x: f'[{x.min():.1f}, {x.max():.1f}]'),
        median_T=('T_joint', 'median')
    ).sort_values('n', ascending=False)
    print(fn_by_fam.head(15).to_string())
    print(f'\nFN by category: {fn["category"].value_counts().to_string()}')
    print(f'FN by spread_pattern: {fn["spread_pattern"].value_counts().to_string()}')

In [ ]:
# ── Misclassification: original scatterplots ──
# Load raw x-y points
pts_main = np.load(S1_DIR / 'scatter_points.npz')
pts_null = np.load(S1_DIR / 'null_expanded_points.npz')
x_main, y_main = pts_main['x'], pts_main['y']
x_null, y_null = pts_null['x'], pts_null['y']
offset = cases_main['case_id'].max()  # 91136

def get_xy(case_id):
    if case_id <= offset:
        idx = case_id - 1
        return x_main[idx], y_main[idx]
    else:
        idx = case_id - offset - 1
        return x_null[idx], y_null[idx]

fp = df[(df['category'].isin(null_cats)) & (df['classification'] == 'detectable')]
fn = df[(df['category'].isin(signal_cats)) & (df['classification'] == 'not_detectable')]

# ── False Positives: sample up to 16 ──
fp_sample = fp.sample(n=min(16, len(fp)), random_state=42) if len(fp) > 0 else fp
n_fp = len(fp_sample)

if n_fp > 0:
    ncols = 4
    nrows = int(np.ceil(n_fp / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
    axes = np.atleast_2d(axes)
    fig.suptitle(f'False Positives — true_null misclassified as detectable  '
                 f'(showing {n_fp} / {len(fp)})',
                 fontsize=13, fontweight='bold', y=1.02)
    for i, (_, row) in enumerate(fp_sample.iterrows()):
        ax = axes[i // ncols, i % ncols]
        x, y = get_xy(row['case_id'])
        ax.scatter(x, y, s=5, alpha=0.4, color='#e74c3c', edgecolors='none')
        ax.set_title(f"id={row['case_id']}  p={row['p_value']:.4f}\n"
                     f"T={row['T_joint']:.1f}  {row.get('spread_pattern','')}",
                     fontsize=8)
        ax.tick_params(labelsize=7)
    for j in range(i + 1, nrows * ncols):
        axes[j // ncols, j % ncols].axis('off')
    plt.tight_layout()
    plt.show()

# ── False Negatives: sample up to 16 ──
fn_sample = fn.sample(n=min(16, len(fn)), random_state=42) if len(fn) > 0 else fn
n_fn = len(fn_sample)

if n_fn > 0:
    ncols = 4
    nrows = int(np.ceil(n_fn / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
    axes = np.atleast_2d(axes)
    fig.suptitle(f'False Negatives — signal misclassified as not_detectable  '
                 f'(showing {n_fn} / {len(fn)})',
                 fontsize=13, fontweight='bold', y=1.02)
    for i, (_, row) in enumerate(fn_sample.iterrows()):
        ax = axes[i // ncols, i % ncols]
        x, y = get_xy(row['case_id'])
        ax.scatter(x, y, s=5, alpha=0.4, color='#3498db', edgecolors='none')
        cat_short = row['category'].replace('variance_only','var').replace('mean_only','mean').replace('mean+variance','m+v')
        ax.set_title(f"id={row['case_id']}  p={row['p_value']:.4f}\n"
                     f"{cat_short}  snr={row.get('snr','?')}  {row.get('spread_pattern','')}",
                     fontsize=8)
        ax.tick_params(labelsize=7)
    for j in range(i + 1, nrows * ncols):
        axes[j // ncols, j % ncols].axis('off')
    plt.tight_layout()
    plt.show()

print(f'Total FP: {len(fp)} | Total FN: {len(fn)}')

In [ ]:
# ── Final summary ──
print('=' * 60)
print('PERMUTATION TEST ANALYSIS SUMMARY')
print('=' * 60)
print(f'Total cases:        {len(df):,}')
print(f'Z-score metrics:    {len(z_cols)}')
print(f'Detectable:         {(df["classification"]=="detectable").sum():,} ({(df["classification"]=="detectable").mean():.1%})')
print(f'Not detectable:     {(df["classification"]=="not_detectable").sum():,} ({(df["classification"]=="not_detectable").mean():.1%})')
print(f'Uncertain:          {(df["classification"]=="uncertain").sum():,} ({(df["classification"]=="uncertain").mean():.1%})')
print(f'\nFalse positive rate: {fpr:.2%} (null → detectable)')
print(f'False negative rate: {fnr:.2%} (signal → not_detectable)')
print(f'\nTop driving metric: {top_drivers.index[0]} ({top_drivers.iloc[0]/len(det):.1%} of detectable cases)')
print(f'Highly correlated metric pairs (|r|>0.95): {len(high_corr)}')